In [8]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from typing import List, Tuple

# ---------- Read SExtractor Catalog ----------
def _get_column_names(read_line_object: List[str]) -> List[str]:
    header = [line.split()[2] for line in read_line_object if line.startswith('#')]
    return header

def _get_rows(read_line_object: List[str]) -> List[List[float]]:
    data = [list(map(float, line.split())) for line in read_line_object if not line.startswith('#')]
    return data

def split_names_and_data(read_line_object: List[str]) -> Tuple[List[str], List[List[float]]]:
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> pd.DataFrame:
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    return pd.DataFrame(data, columns=column_names)

# ---------- Find Closest DECam Object ----------
def find_closest_objects(decam_df: pd.DataFrame, target_coords: List[Tuple[str, str]]) -> pd.DataFrame:
    decam_coords = SkyCoord(ra=decam_df['ALPHAPEAK_J2000'].values * u.deg,
                            dec=decam_df['DELTAPEAK_J2000'].values * u.deg)

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)
        closest_obj = decam_df.iloc[idx]
        results.append({
            'Target': f'LAE {i+1}',
            'Input_RA': target.ra.deg,
            'Input_Dec': target.dec.deg,
            'Matched_RA': closest_obj['ALPHAPEAK_J2000'],
            'Matched_Dec': closest_obj['DELTAPEAK_J2000'],
            'Separation_arcsec': sep2d.arcsecond
        })
    return pd.DataFrame(results)

# ---------- Example Usage ----------
if __name__ == '__main__':
    # Path to your DECam .cat catalog
    cat_path = '/Users/aishwarya/Documents/Lyman_alpha/CAT/n964_band_new.cat'  # change as needed

    # Input coordinates in sexagesimal (RA in HH:MM:SS.SSS, Dec in DD:MM:SS.SS)
    input_targets = [
        ("23:48:17.08", "-31:30:18.94 "),
        ("23:52:04.95", "-31:16:20.60"),
    ]

    decam_df = read_cat(cat_path)
    closest_df = find_closest_objects(decam_df, input_targets)

    print("\nClosest DECam Detections:")
    print(closest_df.to_string(index=False))



Closest DECam Detections:
Target   Input_RA  Input_Dec  Matched_RA  Matched_Dec     Separation_arcsec
 LAE 1 357.071167 -31.505261  357.071139   -31.505258 [0.08490955254462808]
 LAE 2 358.020625 -31.272389  358.020614   -31.272371 [0.07275262468618995]


In [9]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from typing import List, Tuple

# ---------- Star Selection using Pan-STARRS ----------
def is_star_ps1(ps_df: pd.DataFrame, band: str = 'i', threshold: float = 0.05) -> np.ndarray:
    """
    Identify stars using PSF - Kron magnitude difference in Pan-STARRS.
    """
    psf_col = f"{band}PSFMag"
    kron_col = f"{band}KronMag"
    delta_mag = ps_df[psf_col] - ps_df[kron_col]
    return np.abs(delta_mag) < threshold

# ---------- Read Catalogs ----------
def read_decam_catalog(filename: str) -> pd.DataFrame:
    with open(filename) as f:
        header = f.readline().strip().split(',')
    return pd.read_csv(filename, skiprows=1, names=header)

def read_ps_catalog(filename: str) -> pd.DataFrame:
    return pd.read_csv(filename)

# ---------- Find Closest DECam Objects to Input Targets ----------
def find_closest_objects(decam_df: pd.DataFrame, target_coords: List[Tuple[str, str]]) -> pd.DataFrame:
    """
    Match target coordinates (RA, Dec in sexagesimal) to the closest DECam detections.
    """
    decam_coords = SkyCoord(ra=decam_df['ALPHAPEAK_J2000'].values * u.deg,
                            dec=decam_df['DELTAPEAK_J2000'].values * u.deg)

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)
        closest_obj = decam_df.iloc[idx]
        results.append({
            'Target': f'Target {i+1}',
            'Input_RA': target.ra.deg,
            'Input_Dec': target.dec.deg,
            'Matched_RA': closest_obj['ALPHAPEAK_J2000'],
            'Matched_Dec': closest_obj['DELTAPEAK_J2000'],
            'Separation_arcsec': sep2d.arcsecond
        })
    return pd.DataFrame(results)

# ---------- Main ----------
if __name__ == '__main__':
    # Input file paths
    decam_path = "/Users/aishwarya/Documents/Lyman_alpha/CAT/New_cat/i_band_depth_decam_matched.cat"
    ps_path = "/Users/aishwarya/Documents/Lyman_alpha/CAT/New_cat/i_band_depth_panstarrs_matched.csv"

    # Target coordinates (RA, Dec in sexagesimal)
    input_targets = [
        ("23:48:17.338", "-31:31:06.30"),
        ("23:52:02.180", "-31:15:59.70"),
    ]

    # Read catalogs
    decam_df = read_decam_catalog(decam_path)
    ps_df = read_ps_catalog(ps_path)

    # Sanity check ;)
    assert len(decam_df) == len(ps_df), "Catalogs are not matched 1:1 in row count!"

    # Select only star-like objects from Pan-STARRS
    star_mask = is_star_ps1(ps_df, band='z', threshold=0.05)

    # Apply mask to DECam catalog
    decam_stars_df = decam_df[star_mask].reset_index(drop=True)

    # Match targets to star-only DECam catalog
    closest_df = find_closest_objects(decam_stars_df, input_targets)

    print("\nClosest DECam Detections (from star-selected catalog i):")
    print(closest_df.to_string(index=False))


KeyError: 'ALPHAPEAK_J2000'

In [2]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from typing import List, Tuple

# ---------- Read SExtractor Catalog ----------
def _get_column_names(read_line_object: List[str]) -> List[str]:
    header = [line.split()[2] for line in read_line_object if line.startswith('#')]
    return header

def _get_rows(read_line_object: List[str]) -> List[List[float]]:
    data = [list(map(float, line.split())) for line in read_line_object if not line.startswith('#')]
    return data

def split_names_and_data(read_line_object: List[str]) -> Tuple[List[str], List[List[float]]]:
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> pd.DataFrame:
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    return pd.DataFrame(data, columns=column_names)

# ---------- Find Closest DECam Object ----------
def find_closest_objects(decam_df: pd.DataFrame, target_coords: List[Tuple[str, str]]) -> pd.DataFrame:
    decam_coords = SkyCoord(ra=decam_df['ALPHA_J2000'].values * u.deg,
                            dec=decam_df['DELTA_J2000'].values * u.deg)

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)
        closest_obj = decam_df.iloc[idx]
        results.append({
            'Target': f'LAE {i+1}',
            'Input_RA': target.ra.deg,
            'Input_Dec': target.dec.deg,
            'Matched_RA': closest_obj['ALPHA_J2000'],
            'Matched_Dec': closest_obj['DELTA_J2000'],
            'Separation_arcsec': sep2d.arcsecond
        })
    return pd.DataFrame(results)

# ---------- Example Usage ----------
if __name__ == '__main__':
    # Path to your DECam .cat catalog
    cat_path = '/Users/aishwarya/Documents/Lyman_alpha/Mosaiced_images/z_band.cat'  # change as needed

    # Input coordinates in sexagesimal (RA in HH:MM:SS.SSS, Dec in DD:MM:SS.SS)
    # Input coordinates in sexagesimal (RA in HH:MM:SS.SSS, Dec in DD:MM:SS.SS)
input_targets = [
    ("23:48:33.33", "-30:54:10.23"),  # QSO
    ("23:50:39.46", "-31:47:26.41"),  # LAE-1
    ("23:49:25.39", "-31:46:34.95"),  # LAE-2
    ("23:49:45.53", "-31:39:01.46"),  # LAE-3
    ("23:48:22.23", "-31:37:53.92"),  # LAE-4
    ("23:48:17.08", "-31:30:18.94"),  # LAE-5
    ("23:46:51.65", "-31:19:29.93"),  # LAE-6
    ("23:52:04.95", "-31:16:20.60"),  # LAE-7
    ("23:44:51.34", "-31:14:01.78"),  # LAE-8
    ("23:49:09.11", "-31:11:23.68"),  # LAE-9
    ("23:48:38.18", "-31:10:25.49"),  # LAE-10
    ("23:52:11.44", "-31:10:09.82"),  # LAE-11
    ("23:50:09.67", "-31:00:28.43"),  # LAE-12
    ("23:52:50.08", "-30:59:23.05"),  # LAE-13
    ("23:46:52.33", "-30:57:07.02"),  # LAE-14
    ("23:52:39.27", "-30:51:50.73"),  # LAE-15
    ("23:51:42.32", "-30:50:46.97"),  # LAE-16
    ("23:45:30.76", "-30:50:42.20"),  # LAE-17
    ("23:45:38.56", "-30:41:06.23"),  # LAE-18
    ("23:50:56.96", "-30:40:16.95"),  # LAE-19
    ("23:46:20.25", "-30:40:03.94"),  # LAE-20
    ("23:48:12.27", "-30:32:34.32"),  # LAE-21
    ("23:44:45.07", "-30:31:07.12"),  # LAE-22
    ("23:47:37.50", "-30:28:28.35"),  # LAE-23
    ("23:44:31.64", "-30:27:35.79"),  # LAE-24
    ("23:47:38.55", "-30:25:47.75"),  # LAE-25
    ("23:44:36.91", "-30:25:04.43"),  # LAE-26
    ("23:48:21.02", "-30:24:04.19"),  # LAE-27
    ("23:47:21.19", "-30:22:53.54"),  # LAE-28
    ("23:46:27.28", "-30:21:51.57"),  # LAE-29
    ("23:49:27.05", "-30:21:25.98"),  # LAE-30
    ("23:50:27.20", "-30:21:13.39"),  # LAE-31
    ("23:45:02.91", "-30:12:32.87"),  # LAE-32
    ("23:49:34.65", "-30:12:40.49"),  # LAE-33
    ("23:50:23.24", "-30:10:45.41"),  # LAE-34
    ("23:50:45.07", "-30:05:13.35"),  # LAE-35
    ("23:49:43.48", "-30:03:01.26"),  # LAE-36
    ("23:50:36.54", "-30:01:46.79"),  # LAE-37
    ("23:48:43.58", "-29:53:13.94"),  # LAE-38
]


decam_df = read_cat(cat_path)
closest_df = find_closest_objects(decam_df, input_targets)

print("\nClosest DECam Detections:")
print(closest_df.to_string(index=False))



Closest DECam Detections:
Target   Input_RA  Input_Dec  Matched_RA  Matched_Dec    Separation_arcsec
 LAE 1 357.138875 -30.902842  357.130361   -31.650511 [2691.7363429131124]
 LAE 2 357.664417 -31.790669  357.663918   -31.807564 [60.840629397592515]
 LAE 3 357.355792 -31.776375  357.351087   -31.806041 [107.76513989607851]
 LAE 4 357.439708 -31.650406  357.323643   -31.655395 [356.13359079903194]
 LAE 5 357.092625 -31.631644  357.091522   -31.649924  [65.89250255078356]
 LAE 6 357.071167 -31.505261  357.071214   -31.649593  [519.5941000231635]
 LAE 7 356.715208 -31.324981  356.419213   -31.467605 [1044.4800350206253]
 LAE 8 358.020625 -31.272389  358.045066   -31.318364 [181.78681808950566]
 LAE 9 356.213917 -31.233828  356.212362   -31.305275 [257.25448798060967]
LAE 10 357.287958 -31.189911  357.280895   -31.652758 [1666.3915396844268]
LAE 11 357.159083 -31.173747  357.146346   -31.650736 [1717.6051278755367]
LAE 12 358.047667 -31.169394  358.213762   -31.173388 [511.81478244552477

In [66]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from typing import List, Tuple

# ---------- Read SExtractor Catalog ----------
def _get_column_names(read_line_object: List[str]) -> List[str]:
    return [line.split()[2] for line in read_line_object if line.startswith('#')]

def _get_rows(read_line_object: List[str]) -> List[List[float]]:
    return [list(map(float, line.split())) for line in read_line_object if not line.startswith('#')]

def split_names_and_data(read_line_object: List[str]) -> Tuple[List[str], List[List[float]]]:
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> pd.DataFrame:
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    return pd.DataFrame(data, columns=column_names)

# ---------- Find Closest DECam Object ----------
def find_closest_objects(decam_df: pd.DataFrame, target_coords: List[Tuple[str, str]]) -> pd.DataFrame:
    # Use ALPHA_J2000 and DELTA_J2000 from your catalog
    decam_coords = SkyCoord(ra=decam_df['ALPHA_J2000'].values * u.deg,
                            dec=decam_df['DELTA_J2000'].values * u.deg)

    target_names = ["QSO"] + [f"LAE-{i}" for i in range(1, len(target_coords))]

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)
        closest_obj = decam_df.iloc[idx]

        results.append({
            'Target': target_names[i],
            'Input_RA_deg': target.ra.deg,
            'Input_Dec_deg': target.dec.deg,
            'Matched_RA_deg': closest_obj['ALPHA_J2000'],
            'Matched_Dec_deg': closest_obj['DELTA_J2000'],
            'Separation_arcsec': sep2d.arcsecond,
            'MAG_AUTO': closest_obj['MAG_AUTO'],
            'MAGERR_AUTO': closest_obj['MAGERR_AUTO']
        })
    return pd.DataFrame(results)

# ---------- Example Usage ----------
if __name__ == '__main__':
    cat_path = '/Users/aishwarya/Documents/Lyman_alpha/CAT/n964_band.cat'  

    input_targets = [
        ("23:48:33.33", "-30:54:10.23"),  # QSO
        ("23:50:39.46", "-31:47:26.41"),  # LAE-1
        ("23:49:25.39", "-31:46:34.95"),  # LAE-2
        ("23:49:45.53", "-31:39:01.46"),  # LAE-3
        ("23:48:22.23", "-31:37:53.92"),  # LAE-4
        ("23:48:17.08", "-31:30:18.94"),  # LAE-5
        ("23:46:51.65", "-31:19:29.93"),  # LAE-6
        ("23:52:04.95", "-31:16:20.60"),  # LAE-7
        ("23:44:51.34", "-31:14:01.78"),  # LAE-8
        ("23:49:09.11", "-31:11:23.68"),  # LAE-9
        ("23:48:38.18", "-31:10:25.49"),  # LAE-10
        ("23:52:11.44", "-31:10:09.82"),  # LAE-11
        ("23:50:09.67", "-31:00:28.43"),  # LAE-12
        ("23:52:50.08", "-30:59:23.05"),  # LAE-13
        ("23:46:52.33", "-30:57:07.02"),  # LAE-14
        ("23:52:39.27", "-30:51:50.73"),  # LAE-15
        ("23:51:42.32", "-30:50:46.97"),  # LAE-16
        ("23:45:30.76", "-30:50:42.20"),  # LAE-17
        ("23:45:38.56", "-30:41:06.23"),  # LAE-18
        ("23:50:56.96", "-30:40:16.95"),  # LAE-19
        ("23:46:20.25", "-30:40:03.94"),  # LAE-20
        ("23:48:12.27", "-30:32:34.32"),  # LAE-21
        ("23:44:45.07", "-30:31:07.12"),  # LAE-22
        ("23:47:37.50", "-30:28:28.35"),  # LAE-23
        ("23:44:31.64", "-30:27:35.79"),  # LAE-24
        ("23:47:38.55", "-30:25:47.75"),  # LAE-25
        ("23:44:36.91", "-30:25:04.43"),  # LAE-26
        ("23:48:21.02", "-30:24:04.19"),  # LAE-27
        ("23:47:21.19", "-30:22:53.54"),  # LAE-28
        ("23:46:27.28", "-30:21:51.57"),  # LAE-29
        ("23:49:27.05", "-30:21:25.98"),  # LAE-30
        ("23:50:27.20", "-30:21:13.39"),  # LAE-31
        ("23:45:02.91", "-30:12:32.87"),  # LAE-32
        ("23:49:34.65", "-30:12:40.49"),  # LAE-33
        ("23:50:23.24", "-30:10:45.41"),  # LAE-34
        ("23:50:45.07", "-30:05:13.35"),  # LAE-35
        ("23:49:43.48", "-30:03:01.26"),  # LAE-36
        ("23:50:36.54", "-30:01:46.79"),  # LAE-37
        ("23:48:43.58", "-29:53:13.94"),  # LAE-38
    ]

    decam_df = read_cat(cat_path)
    closest_df = find_closest_objects(decam_df, input_targets)

    # Show the first few matches
    print("\nClosest DECam Detections:")
    print(closest_df.head().to_string(index=False))



Closest DECam Detections:
Target  Input_RA_deg  Input_Dec_deg  Matched_RA_deg  Matched_Dec_deg     Separation_arcsec  MAG_AUTO  MAGERR_AUTO
   QSO    357.138875     -30.902842      357.138887       -30.902806 [0.13330151386840192]   -8.5911       0.0091
 LAE-1    357.664417     -31.790669      357.664410       -31.790714  [0.1603426430791098]   -3.7272       0.1754
 LAE-2    357.355792     -31.776375      357.355720       -31.776362  [0.2247163198004731]   -4.8712       0.2175
 LAE-3    357.439708     -31.650406      357.439730       -31.650403 [0.06628504425291978]   -4.2632       0.2003
 LAE-4    357.092625     -31.631644      357.092647       -31.631651 [0.07243917289037614]   -4.2790       0.1849


In [ ]:
A=-8.5911 + 29.01078
print(A)

20.41968


In [67]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from typing import List, Tuple

# ---------- Read SExtractor Catalog ----------
def _get_column_names(read_line_object: List[str]) -> List[str]:
    return [line.split()[2] for line in read_line_object if line.startswith('#')]

def _get_rows(read_line_object: List[str]) -> List[List[float]]:
    return [list(map(float, line.split())) for line in read_line_object if not line.startswith('#')]

def split_names_and_data(read_line_object: List[str]) -> Tuple[List[str], List[List[float]]]:
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> pd.DataFrame:
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    df = pd.DataFrame(data, columns=column_names)
    print(f"Loaded catalog with columns: {df.columns.tolist()}")
    return df

# ---------- Find Closest DECam Object ----------
def find_closest_objects(
    decam_df: pd.DataFrame,
    target_coords: List[Tuple[str, str]],
    max_sep_arcsec: float = 1.0
) -> pd.DataFrame:
    # Ensure required columns exist
    required_cols = ['ALPHA_J2000', 'DELTA_J2000', 'MAG_AUTO', 'MAGERR_AUTO']
    for col in required_cols:
        if col not in decam_df.columns:
            raise KeyError(f"Column '{col}' not found in SExtractor catalog!")

    # SkyCoord for all catalog detections
    decam_coords = SkyCoord(
        ra=decam_df['ALPHA_J2000'].values * u.deg,
        dec=decam_df['DELTA_J2000'].values * u.deg
    )

    # Generate target names
    target_names = ["QSO"] + [f"LAE-{i}" for i in range(1, len(target_coords))]

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)

        # If separation > max allowed, mark as non-detected
        if sep2d.arcsec > max_sep_arcsec:
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': np.nan,
                'Matched_Dec_deg': np.nan,
                'Separation_arcsec': np.nan,
                'MAG_AUTO': np.nan,
                'MAGERR_AUTO': np.nan,
                'Detected': False
            })
        else:
            closest_obj = decam_df.iloc[idx]
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': closest_obj['ALPHA_J2000'],
                'Matched_Dec_deg': closest_obj['DELTA_J2000'],
                'Separation_arcsec': sep2d.arcsec,
                'MAG_AUTO': closest_obj['MAG_AUTO'],
                'MAGERR_AUTO': closest_obj['MAGERR_AUTO'],
                'Detected': True
            })

    return pd.DataFrame(results)

# ---------- Example Usage ----------
if __name__ == '__main__':
    cat_path = '/Users/aishwarya/Documents/Lyman_alpha/CAT/n964_band.cat'  

    input_targets = [
        ("23:48:33.33", "-30:54:10.23"),  # QSO
        ("23:50:39.46", "-31:47:26.41"),  # LAE-1
        ("23:49:25.39", "-31:46:34.95"),  # LAE-2
        ("23:49:45.53", "-31:39:01.46"),  # LAE-3
        ("23:48:22.23", "-31:37:53.92"),  # LAE-4
        ("23:48:17.08", "-31:30:18.94"),  # LAE-5
        ("23:46:51.65", "-31:19:29.93"),  # LAE-6
        ("23:52:04.95", "-31:16:20.60"),  # LAE-7
        ("23:44:51.34", "-31:14:01.78"),  # LAE-8
        ("23:49:09.11", "-31:11:23.68"),  # LAE-9
        ("23:48:38.18", "-31:10:25.49"),  # LAE-10
        ("23:52:11.44", "-31:10:09.82"),  # LAE-11
        ("23:50:09.67", "-31:00:28.43"),  # LAE-12
        ("23:52:50.08", "-30:59:23.05"),  # LAE-13
        ("23:46:52.33", "-30:57:07.02"),  # LAE-14
        ("23:52:39.27", "-30:51:50.73"),  # LAE-15
        ("23:51:42.32", "-30:50:46.97"),  # LAE-16
        ("23:45:30.76", "-30:50:42.20"),  # LAE-17
        ("23:45:38.56", "-30:41:06.23"),  # LAE-18
        ("23:50:56.96", "-30:40:16.95"),  # LAE-19
        ("23:46:20.25", "-30:40:03.94"),  # LAE-20
        ("23:48:12.27", "-30:32:34.32"),  # LAE-21
        ("23:44:45.07", "-30:31:07.12"),  # LAE-22
        ("23:47:37.50", "-30:28:28.35"),  # LAE-23
        ("23:44:31.64", "-30:27:35.79"),  # LAE-24
        ("23:47:38.55", "-30:25:47.75"),  # LAE-25
        ("23:44:36.91", "-30:25:04.43"),  # LAE-26
        ("23:48:21.02", "-30:24:04.19"),  # LAE-27
        ("23:47:21.19", "-30:22:53.54"),  # LAE-28
        ("23:46:27.28", "-30:21:51.57"),  # LAE-29
        ("23:49:27.05", "-30:21:25.98"),  # LAE-30
        ("23:50:27.20", "-30:21:13.39"),  # LAE-31
        ("23:45:02.91", "-30:12:32.87"),  # LAE-32
        ("23:49:34.65", "-30:12:40.49"),  # LAE-33
        ("23:50:23.24", "-30:10:45.41"),  # LAE-34
        ("23:50:45.07", "-30:05:13.35"),  # LAE-35
        ("23:49:43.48", "-30:03:01.26"),  # LAE-36
        ("23:50:36.54", "-30:01:46.79"),  # LAE-37
        ("23:48:43.58", "-29:53:13.94"),  # LAE-38
    ]

    decam_df = read_cat(cat_path)
    closest_df = find_closest_objects(decam_df, input_targets, max_sep_arcsec=1.0)

    # Show all matches
    print("\nClosest DECam Detections:")
    print(closest_df.to_string(index=False))

    # Save to CSV for inspection
    closest_df.to_csv('n964_matched_targets.csv', index=False)
    print("\nSaved results to n964_matched_targets.csv")


Loaded catalog with columns: ['NUMBER', 'X_IMAGE', 'Y_IMAGE', 'ALPHA_J2000', 'DELTA_J2000', 'MAG_APER', 'MAGERR_APER', 'MAG_AUTO', 'MAGERR_AUTO', 'FLAGS']

Closest DECam Detections:
Target  Input_RA_deg  Input_Dec_deg  Matched_RA_deg  Matched_Dec_deg      Separation_arcsec  MAG_AUTO  MAGERR_AUTO  Detected
   QSO    357.138875     -30.902842      357.138887       -30.902806  [0.13330151386840192]   -8.5911       0.0091      True
 LAE-1    357.664417     -31.790669      357.664410       -31.790714   [0.1603426430791098]   -3.7272       0.1754      True
 LAE-2    357.355792     -31.776375      357.355720       -31.776362   [0.2247163198004731]   -4.8712       0.2175      True
 LAE-3    357.439708     -31.650406      357.439730       -31.650403  [0.06628504425291978]   -4.2632       0.2003      True
 LAE-4    357.092625     -31.631644      357.092647       -31.631651  [0.07243917289037614]   -4.2790       0.1849      True
 LAE-5    357.071167     -31.505261      357.071173       -31.505304

In [6]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from typing import List, Tuple

# ---------- Read SExtractor Catalog ----------
def _get_column_names(read_line_object: List[str]) -> List[str]:
    return [line.split()[2] for line in read_line_object if line.startswith('#')]

def _get_rows(read_line_object: List[str]) -> List[List[float]]:
    return [list(map(float, line.split())) for line in read_line_object if not line.startswith('#')]

def split_names_and_data(read_line_object: List[str]) -> Tuple[List[str], List[List[float]]]:
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> pd.DataFrame:
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    df = pd.DataFrame(data, columns=column_names)
    print(f"Loaded catalog with columns: {df.columns.tolist()}")
    return df

# ---------- Find Closest DECam Object ----------
def find_closest_objects(
    decam_df: pd.DataFrame,
    target_coords: List[Tuple[str, str]],
    max_sep_arcsec: float = 1.0
) -> pd.DataFrame:
    # Ensure required columns exist
    required_cols = ['ALPHA_J2000', 'DELTA_J2000', 'MAG_AUTO', 'MAGERR_AUTO']
    for col in required_cols:
        if col not in decam_df.columns:
            raise KeyError(f"Column '{col}' not found in SExtractor catalog!")

    # SkyCoord for all catalog detections
    decam_coords = SkyCoord(
        ra=decam_df['ALPHA_J2000'].values * u.deg,
        dec=decam_df['DELTA_J2000'].values * u.deg
    )

    # Generate target names
    target_names = ["QSO"] + [f"LAE-{i}" for i in range(1, len(target_coords))]

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)

        # If separation > max allowed, mark as non-detected
        if sep2d.arcsec > max_sep_arcsec:
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': np.nan,
                'Matched_Dec_deg': np.nan,
                'Separation_arcsec': np.nan,
                'MAG_AUTO': np.nan,
                'MAGERR_AUTO': np.nan,
                'Detected': False
            })
        else:
            closest_obj = decam_df.iloc[idx]
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': closest_obj['ALPHA_J2000'],
                'Matched_Dec_deg': closest_obj['DELTA_J2000'],
                'Separation_arcsec': sep2d.arcsec,
                'MAG_APER': closest_obj['MAG_APER']+30.243,  # Adjusted MAG_APER
                'MAGERR_APER': closest_obj['MAGERR_APER'],
                'Detected': True
            })

    return pd.DataFrame(results)

# ---------- Example Usage ----------
if __name__ == '__main__':
    cat_path = '/Users/aishwarya/Documents/Lyman_alpha/CAT/i_band.cat'  

    input_targets = [
        ("23:48:33.33", "-30:54:10.23"),  # QSO
        ("23:50:39.46", "-31:47:26.41"),  # LAE-1
        ("23:49:25.39", "-31:46:34.95"),  # LAE-2
        ("23:49:45.53", "-31:39:01.46"),  # LAE-3
        ("23:48:22.23", "-31:37:53.92"),  # LAE-4
        ("23:48:17.08", "-31:30:18.94"),  # LAE-5
        ("23:46:51.65", "-31:19:29.93"),  # LAE-6
        ("23:52:04.95", "-31:16:20.60"),  # LAE-7
        ("23:44:51.34", "-31:14:01.78"),  # LAE-8
        ("23:49:09.11", "-31:11:23.68"),  # LAE-9
        ("23:48:38.18", "-31:10:25.49"),  # LAE-10
        ("23:52:11.44", "-31:10:09.82"),  # LAE-11
        ("23:50:09.67", "-31:00:28.43"),  # LAE-12
        ("23:52:50.08", "-30:59:23.05"),  # LAE-13
        ("23:46:52.33", "-30:57:07.02"),  # LAE-14
        ("23:52:39.27", "-30:51:50.73"),  # LAE-15
        ("23:51:42.32", "-30:50:46.97"),  # LAE-16
        ("23:45:30.76", "-30:50:42.20"),  # LAE-17
        ("23:45:38.56", "-30:41:06.23"),  # LAE-18
        ("23:50:56.96", "-30:40:16.95"),  # LAE-19
        ("23:46:20.25", "-30:40:03.94"),  # LAE-20
        ("23:48:12.27", "-30:32:34.32"),  # LAE-21
        ("23:44:45.07", "-30:31:07.12"),  # LAE-22
        ("23:47:37.50", "-30:28:28.35"),  # LAE-23
        ("23:44:31.64", "-30:27:35.79"),  # LAE-24
        ("23:47:38.55", "-30:25:47.75"),  # LAE-25
        ("23:44:36.91", "-30:25:04.43"),  # LAE-26
        ("23:48:21.02", "-30:24:04.19"),  # LAE-27
        ("23:47:21.19", "-30:22:53.54"),  # LAE-28
        ("23:46:27.28", "-30:21:51.57"),  # LAE-29
        ("23:49:27.05", "-30:21:25.98"),  # LAE-30
        ("23:50:27.20", "-30:21:13.39"),  # LAE-31
        ("23:45:02.91", "-30:12:32.87"),  # LAE-32
        ("23:49:34.65", "-30:12:40.49"),  # LAE-33
        ("23:50:23.24", "-30:10:45.41"),  # LAE-34
        ("23:50:45.07", "-30:05:13.35"),  # LAE-35
        ("23:49:43.48", "-30:03:01.26"),  # LAE-36
        ("23:50:36.54", "-30:01:46.79"),  # LAE-37
        ("23:48:43.58", "-29:53:13.94"),  # LAE-38
    ]

    decam_df = read_cat(cat_path)
    closest_df = find_closest_objects(decam_df, input_targets, max_sep_arcsec=1.0)

    # Show all matches
    print("\nClosest DECam Detections:")
    print(closest_df.to_string(index=False))
    # Count True detections
    num_detected = closest_df['Detected'].sum()
    print(f"Number of true detections: {num_detected}")

    # Save to CSV for inspection
    closest_df.to_csv('n964_matched_targets.csv', index=False)
    print (len(closest_df), "matches found.")
    print("\nSaved results to n964_matched_targets.csv")


Loaded catalog with columns: ['NUMBER', 'X_IMAGE', 'Y_IMAGE', 'ALPHA_J2000', 'DELTA_J2000', 'MAG_APER', 'MAGERR_APER', 'MAG_AUTO', 'MAGERR_AUTO', 'FLAGS']

Closest DECam Detections:
Target  Input_RA_deg  Input_Dec_deg  Matched_RA_deg  Matched_Dec_deg     Separation_arcsec  MAG_APER  MAGERR_APER  Detected  MAG_AUTO  MAGERR_AUTO
   QSO    357.138875     -30.902842      357.138933       -30.902832 [0.18206778694148362]  129.2430      99.0000      True       NaN          NaN
 LAE-1    357.664417     -31.790669             NaN              NaN                   NaN       NaN          NaN     False       NaN          NaN
 LAE-2    357.355792     -31.776375             NaN              NaN                   NaN       NaN          NaN     False       NaN          NaN
 LAE-3    357.439708     -31.650406             NaN              NaN                   NaN       NaN          NaN     False       NaN          NaN
 LAE-4    357.092625     -31.631644             NaN              NaN               

In [1]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from typing import List, Tuple

# ---------- Read SExtractor Catalog ----------
def _get_column_names(read_line_object: List[str]) -> List[str]:
    return [line.split()[2] for line in read_line_object if line.startswith('#')]

def _get_rows(read_line_object: List[str]) -> List[List[float]]:
    return [list(map(float, line.split())) for line in read_line_object if not line.startswith('#')]

def split_names_and_data(read_line_object: List[str]) -> Tuple[List[str], List[List[float]]]:
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> pd.DataFrame:
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    df = pd.DataFrame(data, columns=column_names)
    print(f"Loaded catalog with columns: {df.columns.tolist()}")
    return df

# ---------- Find Closest DECam Object ----------
def find_closest_objects(
    decam_df: pd.DataFrame,
    target_coords: List[Tuple[str, str]],
    max_sep_arcsec: float = 1.0
) -> pd.DataFrame:
    # Ensure required columns exist
    required_cols = ['ALPHA_J2000', 'DELTA_J2000', 'MAG_AUTO', 'MAGERR_AUTO']
    for col in required_cols:
        if col not in decam_df.columns:
            raise KeyError(f"Column '{col}' not found in SExtractor catalog!")

    # SkyCoord for all catalog detections
    decam_coords = SkyCoord(
        ra=decam_df['ALPHA_J2000'].values * u.deg,
        dec=decam_df['DELTA_J2000'].values * u.deg
    )

    # Generate target names
    target_names = ["QSO"] + [f"LAE-{i}" for i in range(1, len(target_coords))]

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)

        # If separation > max allowed, mark as non-detected
        if sep2d.arcsec > max_sep_arcsec:
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': np.nan,
                'Matched_Dec_deg': np.nan,
                'Separation_arcsec': np.nan,
                'MAG_AUTO': np.nan,
                'MAGERR_AUTO': np.nan,
                'Detected': False
            })
        else:
            closest_obj = decam_df.iloc[idx]
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': closest_obj['ALPHA_J2000'],
                'Matched_Dec_deg': closest_obj['DELTA_J2000'],
                'Separation_arcsec': sep2d.arcsec,
                'MAG_APER': closest_obj['MAG_APER']+30.243,  # Adjusted MAG_APER
                'MAGERR_APER': closest_obj['MAGERR_APER'],
                'Detected': True
            })

    return pd.DataFrame(results)

# ---------- Example Usage ----------
if __name__ == '__main__':
    cat_path = '/Users/aishwarya/Documents/Lyman_alpha/CAT/n964_band.cat'  

    input_targets = [
        ("23:48:33.33", "-30:54:10.23"),  # QSO
        ("23:50:39.46", "-31:47:26.41"),  # LAE-1
        ("23:49:25.39", "-31:46:34.95"),  # LAE-2
        ("23:49:45.53", "-31:39:01.46"),  # LAE-3
        ("23:48:22.23", "-31:37:53.92"),  # LAE-4
        ("23:48:17.08", "-31:30:18.94"),  # LAE-5
        ("23:46:51.65", "-31:19:29.93"),  # LAE-6
        ("23:52:04.95", "-31:16:20.60"),  # LAE-7
        ("23:44:51.34", "-31:14:01.78"),  # LAE-8
        ("23:49:09.11", "-31:11:23.68"),  # LAE-9
        ("23:48:38.18", "-31:10:25.49"),  # LAE-10
        ("23:52:11.44", "-31:10:09.82"),  # LAE-11
        ("23:50:09.67", "-31:00:28.43"),  # LAE-12
        ("23:52:50.08", "-30:59:23.05"),  # LAE-13
        ("23:46:52.33", "-30:57:07.02"),  # LAE-14
        ("23:52:39.27", "-30:51:50.73"),  # LAE-15
        ("23:51:42.32", "-30:50:46.97"),  # LAE-16
        ("23:45:30.76", "-30:50:42.20"),  # LAE-17
        ("23:45:38.56", "-30:41:06.23"),  # LAE-18
        ("23:50:56.96", "-30:40:16.95"),  # LAE-19
        ("23:46:20.25", "-30:40:03.94"),  # LAE-20
        ("23:48:12.27", "-30:32:34.32"),  # LAE-21
        ("23:44:45.07", "-30:31:07.12"),  # LAE-22
        ("23:47:37.50", "-30:28:28.35"),  # LAE-23
        ("23:44:31.64", "-30:27:35.79"),  # LAE-24
        ("23:47:38.55", "-30:25:47.75"),  # LAE-25
        ("23:44:36.91", "-30:25:04.43"),  # LAE-26
        ("23:48:21.02", "-30:24:04.19"),  # LAE-27
        ("23:47:21.19", "-30:22:53.54"),  # LAE-28
        ("23:46:27.28", "-30:21:51.57"),  # LAE-29
        ("23:49:27.05", "-30:21:25.98"),  # LAE-30
        ("23:50:27.20", "-30:21:13.39"),  # LAE-31
        ("23:45:02.91", "-30:12:32.87"),  # LAE-32
        ("23:49:34.65", "-30:12:40.49"),  # LAE-33
        ("23:50:23.24", "-30:10:45.41"),  # LAE-34
        ("23:50:45.07", "-30:05:13.35"),  # LAE-35
        ("23:49:43.48", "-30:03:01.26"),  # LAE-36
        ("23:50:36.54", "-30:01:46.79"),  # LAE-37
        ("23:48:43.58", "-29:53:13.94"),  # LAE-38
    ]

    decam_df = read_cat(cat_path)
    closest_df = find_closest_objects(decam_df, input_targets, max_sep_arcsec=1.0)

    # Show all matches
    print("\nClosest DECam Detections:")
    print(closest_df.to_string(index=False))
    # Count True detections
    num_detected = closest_df['Detected'].sum()
    print(f"Number of true detections: {num_detected}")

    # Save to CSV for inspection
    closest_df.to_csv('n964_matched_targets.csv', index=False)
    print (len(closest_df), "matches found.")
    print("\nSaved results to n964_matched_targets.csv")


Loaded catalog with columns: ['NUMBER', 'X_IMAGE', 'Y_IMAGE', 'ALPHA_J2000', 'DELTA_J2000', 'MAG_APER', 'MAGERR_APER', 'MAG_AUTO', 'MAGERR_AUTO', 'FLAGS']

Closest DECam Detections:
Target  Input_RA_deg  Input_Dec_deg  Matched_RA_deg  Matched_Dec_deg      Separation_arcsec  MAG_APER  MAGERR_APER  Detected
   QSO    357.138875     -30.902842      357.138892       -30.902803  [0.14900389380957654]   22.1043       0.0152      True
 LAE-1    357.664417     -31.790669      357.664510       -31.790710  [0.32151218806928067]   25.4099       0.1367      True
 LAE-2    357.355792     -31.776375      357.355737       -31.776354  [0.18262435222062842]   25.8787       0.2014      True
 LAE-3    357.439708     -31.650406      357.439734       -31.650397  [0.08491351018312576]   25.8497       0.1984      True
 LAE-4    357.092625     -31.631644      357.092635       -31.631626  [0.07262802924003683]   25.7892       0.1933      True
 LAE-5    357.071167     -31.505261      357.071176       -31.505300